# TP03: your first neural network

**MSA-DATB09-03 · Neural Networks & Deep Learning Foundations · Session 3 / 8** · Guillaume Saint-Cirgue

*Estimated time: 80 minutes*

In the course we stacked neurons into layers: a hidden layer draws several lines, an output neuron combines
them, and XOR becomes possible. We derived backpropagation by hand (the chain rule, layer by layer) and saw
that PyTorch's `loss.backward()` computes exactly the same gradients. In this notebook you check all of it in
code: the XOR network, one forward and backward pass by hand, then real networks trained with PyTorch.

**How it works.** Each exercise is a cell with a zone to complete:

```python
### START CODE HERE ###

### END CODE HERE ###
```

Write your code between the two markers, then run the cell: the `assert` lines below the zone check your
answer and print `Exercise validated!` when it is correct. If an assert fails, read its message: it tells
you what to look at.

**Run it** on Google Colab (File > Upload notebook) or locally with the course `requirements.txt`.

## Imports

- `numpy`: arrays, for the networks computed by hand (Parts A and B).
- `matplotlib`: plots of decision boundaries.
- `torch`, `torch.nn`: PyTorch, its tensors with autograd and its layers (Parts B to D).
- `sklearn.datasets`: `make_moons` and `make_circles`, two 2-D datasets that no line can separate.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import make_moons, make_circles

FAST_RUN = os.environ.get("FAST_RUN", "0") == "1"   # True: quick check run
np.random.seed(0)
torch.manual_seed(0)
torch.set_num_threads(2)                              # small models: 2 CPU threads are enough

## Helper functions (run this cell, no need to modify it)

- `plot_boundary(model, X, y, title, ax)` colours the plane with the probability predicted by a PyTorch model
  and draws its decision boundary (probability 0.5).
- `step` and `sigmoid` are the two activations of the previous sessions, in NumPy.

In [ ]:
def step(z):
    return (np.asarray(z) >= 0).astype(int)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def plot_boundary(model, X, y, title="", ax=None):
    show = ax is None
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 4))
    x1, x2 = np.meshgrid(np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 200),
                         np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 200))
    grid = torch.tensor(np.c_[x1.ravel(), x2.ravel()], dtype=torch.float32)
    with torch.no_grad():
        P = torch.sigmoid(model(grid)).numpy().reshape(x1.shape)
    ax.contourf(x1, x2, P, levels=np.linspace(0, 1, 11), cmap="coolwarm_r", alpha=0.3)
    ax.contour(x1, x2, P, levels=[0.5], colors="k")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c="#4338ca", s=12)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c="#d97706", s=12, marker="s")
    ax.set_title(title)
    if show:
        plt.show()

# Part A: the XOR network, with the weights of exercise 1

Reminder of the course: XOR = AND(OR, NAND). Hidden neuron 1 computes OR ($w = (1, 1)$, $b = -0.5$),
hidden neuron 2 computes NAND ($w = (-1, -1)$, $b = 1.5$), the output neuron computes AND of the two
($w = (1, 1)$, $b = -1.5$).

In code, examples are **rows**: `X` has shape `(m, n_in)`, and a layer with weight matrix `W` of shape
`(n_out, n_in)` (one row per neuron) and bias `b` of shape `(n_out,)` computes `X @ W.T + b`, shape
`(m, n_out)`. This is exactly what PyTorch's `nn.Linear` does.

### Exercise 1: a dense layer

Write `dense(X, W, b, activation)` that returns `activation(X @ W.T + b)`. `activation` is a function
(for example `step` or `sigmoid`): you call it like any function.

In [ ]:
def dense(X, W, b, activation):
    ### START CODE HERE ###
    A = None
    ### END CODE HERE ###
    return A


X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])
out = dense(X_xor, np.array([[1, 1]]), np.array([-1.5]), step)
assert out.shape == (4, 1), f"one row per example, one column per neuron: expected (4, 1), got {np.shape(out)}"
assert out.ravel().tolist() == [0, 0, 0, 1], "w = (1, 1), b = -1.5 with a step computes AND"
print("Exercise validated!")

### Exercise 2: the weights of the XOR network

Create `W1` (shape `(2, 2)`: row 0 = the OR neuron, row 1 = the NAND neuron), `b1` (shape `(2,)`), `W2`
(shape `(1, 2)`: the AND neuron) and `b2` (shape `(1,)`), as float arrays.

In [ ]:
### START CODE HERE ###

### END CODE HERE ###

assert W1.shape == (2, 2) and b1.shape == (2,), "layer 1: W1 of shape (2, 2), b1 of shape (2,)"
assert W2.shape == (1, 2) and b2.shape == (1,), "layer 2: W2 of shape (1, 2), b2 of shape (1,)"
assert dense(X_xor, W1[:1], b1[:1], step).ravel().tolist() == [0, 1, 1, 1], "row 0 of W1 (with b1[0]) must compute OR"
assert dense(X_xor, W1[1:], b1[1:], step).ravel().tolist() == [1, 1, 1, 0], "row 1 of W1 (with b1[1]) must compute NAND"
print("Exercise validated!")

### Exercise 3: the forward pass of the network

Write `forward_xor(X, activation)` that returns the hidden layer output `A1` (shape `(m, 2)`) and the network
output `A2` (shape `(m, 1)`), using `dense` twice with the weights above.

In [ ]:
def forward_xor(X, activation=step):
    ### START CODE HERE ###
    A1, A2 = None, None
    ### END CODE HERE ###
    return A1, A2


A1, A2 = forward_xor(X_xor)
print("hidden layer:\n", A1, "\noutput:", A2.ravel())
assert A1.tolist() == [[0, 1], [1, 1], [1, 1], [1, 0]], "the hidden representation of exercise 1 is (0,1), (1,1), (1,1), (1,0)"
assert A2.ravel().tolist() == [0, 1, 1, 0], "the network must compute XOR"
print("Exercise validated!")

**What we see.** The hidden layer maps the 4 inputs to only 3 distinct points, and in this new space one
line (the AND neuron) separates the classes. The hidden layer has built the features that make the problem
easy.

### Exercise 4: the same network with sigmoids

A step has no gradient, so we cannot train it. Replace it by a sigmoid and multiply **all** weights and
biases by 10 (question 2 of exercise 1). Compute `A2_sig`, the output for the 4 inputs (shape `(4, 1)`).

> Hint: you cannot reuse `forward_xor` directly because the weights change; call `dense` twice with
> `10 * W1`, `10 * b1`, and so on.

In [ ]:
### START CODE HERE ###

### END CODE HERE ###

print("outputs:", A2_sig.ravel().round(4))
assert A2_sig.shape == (4, 1), "one output per example"
assert np.isclose(A2_sig[1, 0], 0.9924, atol=1e-3), "for x = (0, 1) the course gives 0.992"
assert np.all(np.abs(A2_sig.ravel() - y_xor) < 0.01), "the sigmoid network should be within 0.01 of XOR"
print("Exercise validated!")

# Part B: one forward and one backward pass by hand

Exercise 3 of the course: a 2-2-1 network with sigmoids and one example $x = (1, 2)$, $y = 1$. For one
example we work with 1-D arrays (the column vectors of the slides): $z^{[1]} = W^{[1]} x + b^{[1]}$ is
`W1 @ x + b1` in NumPy.

In [ ]:
x_ex = np.array([1.0, 2.0])
y_ex = 1.0
W1_ex = np.array([[1.0, -1.0], [0.5, 0.5]])
b1_ex = np.array([1.0, -1.5])
W2_ex = np.array([[1.0, -1.0]])
b2_ex = np.array([0.0])

### Exercise 5: the forward pass

Write `forward_one(x, W1, b1, W2, b2)` that returns `z1, a1, z2, a2`:

$$ z^{[1]} = W^{[1]} x + b^{[1]}, \quad a^{[1]} = \sigma(z^{[1]}), \quad z^{[2]} = W^{[2]} a^{[1]} + b^{[2]}, \quad a^{[2]} = \sigma(z^{[2]}) $$

- `W @ v` with `W` of shape `(k, n)` and `v` of shape `(n,)` gives a vector of shape `(k,)`.

In [ ]:
def forward_one(x, W1, b1, W2, b2):
    ### START CODE HERE ###
    z1, a1, z2, a2 = None, None, None, None
    ### END CODE HERE ###
    return z1, a1, z2, a2


z1, a1, z2, a2 = forward_one(x_ex, W1_ex, b1_ex, W2_ex, b2_ex)
print("z1 =", z1, " a1 =", a1, " z2 =", z2, " a2 =", a2)
assert np.allclose(z1, [0, 0]) and np.allclose(a1, [0.5, 0.5]), "z1 should be (0, 0), so a1 = (0.5, 0.5)"
assert np.shape(a2) == (1,) and np.isclose(a2[0], 0.5), "a2 should be an array of shape (1,) equal to 0.5"
print("Exercise validated!")

### Exercise 6: the loss

Write `loss_one(a2, y)`: the log loss $\mathcal{L} = -[y \log a^{[2]} + (1 - y)\log(1 - a^{[2]})]$ of this
example, returned as a Python number (`float(...)`, since `a2` has shape `(1,)`).

In [ ]:
def loss_one(a2, y):
    ### START CODE HERE ###
    L = None
    ### END CODE HERE ###
    return L


L0 = loss_one(a2, y_ex)
assert isinstance(L0, float), "return a Python float"
assert np.isclose(L0, np.log(2)), "with a2 = 0.5 the loss is log 2 = 0.693"
print("Exercise validated!")

### Exercise 7: the backward pass

Write `backward_one(x, y, W2, a1, a2)` with the formulas of the course:

$$ dz^{[2]} = a^{[2]} - y, \qquad dW^{[2]} = dz^{[2]}\, a^{[1]\top}, \qquad db^{[2]} = dz^{[2]} $$
$$ da^{[1]} = W^{[2]\top} dz^{[2]}, \qquad dz^{[1]} = da^{[1]} \odot a^{[1]} \odot (1 - a^{[1]}), \qquad dW^{[1]} = dz^{[1]}\, x^\top, \qquad db^{[1]} = dz^{[1]} $$

- $u\,v^\top$ for two vectors is their **outer product**, a matrix: `np.outer(u, v)` (shape `(len(u), len(v))`).
  Do not use `u @ v`, which is a dot product (a number).
- `W2.T @ dz2` sends the output error back to the hidden neurons; `*` is the element-wise product $\odot$.

In [ ]:
def backward_one(x, y, W2, a1, a2):
    ### START CODE HERE ###
    dW1, db1, dW2, db2 = None, None, None, None
    ### END CODE HERE ###
    return dW1, db1, dW2, db2


dW1, db1, dW2, db2 = backward_one(x_ex, y_ex, W2_ex, a1, a2)
print("dW2 =", dW2, " db2 =", db2)
print("dW1 =\n", dW1, "\ndb1 =", db1)
assert np.shape(dW2) == W2_ex.shape and np.shape(dW1) == W1_ex.shape, "each gradient has the shape of its parameter"
assert np.allclose(dW2, [[-0.25, -0.25]]) and np.allclose(db2, [-0.5]), "output layer: dW2 = (-0.25, -0.25), db2 = -0.5"
assert np.allclose(dW1, [[-0.125, -0.25], [0.125, 0.25]]), "hidden layer: see the solution of exercise 3"
assert np.allclose(db1, [-0.125, 0.125]), "db1 = dz1 = (-0.125, 0.125)"
print("Exercise validated!")

### Exercise 8: check with finite differences

As in TP02, check two entries of the gradient numerically:

$$ \frac{\partial \mathcal{L}}{\partial W^{[1]}_{11}} \approx \frac{\mathcal{L}(W^{[1]}_{11} + h) - \mathcal{L}(W^{[1]}_{11} - h)}{2h} $$

Compute `num_W1_00` (for `W1_ex[0, 0]`) and `num_W2_01` (for `W2_ex[0, 1]`) with `h = 1e-5`. The helper
`loss_of` computes the loss for given parameters. Work on copies (`W.copy()`): never modify `W1_ex` itself.

In [ ]:
def loss_of(W1, b1, W2, b2):
    return loss_one(forward_one(x_ex, W1, b1, W2, b2)[3], y_ex)


h = 1e-5
### START CODE HERE ###

### END CODE HERE ###

print("numerical:", num_W1_00, num_W2_01, " backprop:", dW1[0, 0], dW2[0, 1])
assert np.allclose(W1_ex, [[1, -1], [0.5, 0.5]]) and np.allclose(W2_ex, [[1, -1]]), "do not modify W1_ex or W2_ex"
assert np.isclose(num_W1_00, dW1[0, 0], atol=1e-8), "the finite difference should match dW1[0, 0]"
assert np.isclose(num_W2_01, dW2[0, 1], atol=1e-8), "the finite difference should match dW2[0, 1]"
print("Exercise validated!")

### Exercise 9: PyTorch's autograd gives the same gradients

Create the four parameters as float64 tensors that require a gradient (`torch.tensor(W1_ex,
requires_grad=True)` keeps the float64 type of the NumPy array), redo the forward pass with
`torch.sigmoid` and the loss with `torch.log`, then call `loss_t.backward()`. Each `param.grad` is then
filled with $\partial \mathcal{L} / \partial \text{param}$.

- The tensor operations are the same as in NumPy: `W1_t @ x_t + b1_t`.
- `loss_t` must be a single number (a 0-dimensional tensor): use `.sum()` at the end, since `a2` has
  shape `(1,)`.

In [ ]:
x_t = torch.tensor(x_ex)
### START CODE HERE ###

### END CODE HERE ###

print("autograd dW1:\n", W1_t.grad, "\nautograd dW2:", W2_t.grad)
assert W1_t.grad is not None, "call loss_t.backward()"
assert np.allclose(W1_t.grad.numpy(), dW1) and np.allclose(b1_t.grad.numpy(), db1), "layer 1: autograd = backprop by hand"
assert np.allclose(W2_t.grad.numpy(), dW2) and np.allclose(b2_t.grad.numpy(), db2), "layer 2: autograd = backprop by hand"
print("Exercise validated! loss.backward() computes exactly the chain rule you wrote.")

### Exercise 10: one gradient step

Update the four NumPy parameters with $\eta = 1$ (`W1_new = W1_ex - eta * dW1`, and so on), then compute the
new loss `L1` with `loss_of`. The course says it drops from 0.693 to 0.285.

In [ ]:
eta = 1.0
### START CODE HERE ###

### END CODE HERE ###

print("new loss:", round(L1, 3))
assert np.allclose(W1_new, [[1.125, -0.75], [0.375, 0.25]]), "check the update of W1"
assert np.isclose(L1, 0.2853, atol=1e-3), "the new loss should be 0.285"
print("Exercise validated!")

**Going further on your own.** Coding a full neural network from scratch in NumPy (forward and backward,
layer by layer, with a training loop) is a great exercise: I do it step by step on my YouTube channel
Machine Learnia, in video 9 of the Deep Learning series (in French):
[Programmer le réseau à 2 couches avec NumPy](https://www.youtube.com/watch?v=LXvSdSKw9dk&list=PLO_fdPEVlfKoanjvTJbIbd9V5d9Pzp8Rw&index=9).
Here we move on to PyTorch, which does the backward pass for us.

# Part C: the network in PyTorch

### Exercise 11: build the network

`nn.Sequential(layer1, layer2, ...)` chains layers: the output of each one is the input of the next.
Build `model` = a 2-3-1 network: `nn.Linear(2, 3)`, then `nn.Sigmoid()`, then `nn.Linear(3, 1)`. No sigmoid
at the end: the loss `BCEWithLogitsLoss` applies it (session 2).

Then count its parameters: `model.parameters()` yields every weight and bias tensor, and `p.numel()` is the
number of elements of tensor `p`. Store the total in `n_params`.

In [ ]:
torch.manual_seed(0)
### START CODE HERE ###

### END CODE HERE ###

print(model)
assert model[0].weight.shape == (3, 2), "the first layer's weight has shape (n_out, n_in) = (3, 2)"
assert isinstance(model[1], nn.Sigmoid) and model[2].weight.shape == (1, 3), "then a sigmoid, then nn.Linear(3, 1)"
assert n_params == 13, "exercise 2 of the course: 6 + 3 + 3 + 1 = 13 parameters"
print("Exercise validated!")

### Exercise 12: the training function

Write `train(model, X_t, y_t, lr, n_epochs)` with the loop of session 2 (full batch: all the examples at
each step):

- create `loss_fn = nn.BCEWithLogitsLoss()` and `optimizer = torch.optim.SGD(model.parameters(), lr=lr)`;
- for each epoch: `optimizer.zero_grad()`, the forward pass `model(X_t).squeeze(1)` (logits of shape `(m,)`),
  the loss, `loss.backward()` (backpropagation through all the layers), `optimizer.step()`;
- append `loss.item()` to the list `losses` and return it.

In [ ]:
def train(model, X_t, y_t, lr=5.0, n_epochs=2000):
    losses = []
    ### START CODE HERE ###

    ### END CODE HERE ###
    return losses


torch.manual_seed(0)
test_model = nn.Sequential(nn.Linear(2, 3), nn.Sigmoid(), nn.Linear(3, 1))
test_losses = train(test_model, torch.rand(8, 2), torch.ones(8), lr=0.5, n_epochs=50)
assert len(test_losses) == 50, "one loss per epoch"
assert test_losses[-1] < test_losses[0], "the loss must decrease"
print("Exercise validated!")

### Exercise 13: accuracy

Write `accuracy(model, X_t, y_t)`: the fraction of correct predictions, as a Python float. A logit $z > 0$
means a probability above 0.5, so class 1.

- `with torch.no_grad():` evaluates without recording the graph (we do not need gradients here).
- `(z > 0).float()` turns booleans into 0.0/1.0; `.mean().item()` gives a Python number.

In [ ]:
def accuracy(model, X_t, y_t):
    ### START CODE HERE ###
    acc = None
    ### END CODE HERE ###
    return acc


assert accuracy(test_model, torch.rand(8, 2), torch.ones(8)) == 1.0, "the test model always predicts 1 by now"
print("Exercise validated!")

### Exercise 14: XOR, learnt

Convert the XOR table to float32 tensors `Xx_t` (shape `(4, 2)`) and `yx_t` (shape `(4,)`), create a fresh
2-3-1 network `model_xor` (after `torch.manual_seed(0)`), train it (`lr=5.0`, 2000 epochs) and compute its
accuracy `acc_xor`.

In [ ]:
### START CODE HERE ###

### END CODE HERE ###

print("final loss:", round(losses_xor[-1], 4), " accuracy:", acc_xor)
assert acc_xor == 1.0, "the network must solve XOR (accuracy 100 %)"
print("Exercise validated! What one neuron could never do, three neurons learnt by themselves.")

### Exercise 15: the moons

The moons of session 2 (logistic regression: 0.885). Train a fresh 2-3-1 network `model_moons` (after
`torch.manual_seed(0)`, `lr=5.0`, 2000 epochs) and compute `acc_moons`.

In [ ]:
X_m, y_m = make_moons(n_samples=200, noise=0.1, random_state=0)
Xm_t, ym_t = torch.tensor(X_m, dtype=torch.float32), torch.tensor(y_m, dtype=torch.float32)

### START CODE HERE ###

### END CODE HERE ###

print("accuracy on the moons:", acc_moons)
assert acc_moons > 0.95, "the network should beat the straight line (0.885) by far"
print("Exercise validated!")

### Exercise 16: the circles

`make_circles` generates a small circle inside a big one (`factor=0.5`: the radius ratio). Same recipe:
`model_circles`, `acc_circles`. Then the cell plots both boundaries.

In [ ]:
X_c, y_c = make_circles(n_samples=200, noise=0.05, factor=0.5, random_state=0)
Xc_t, yc_t = torch.tensor(X_c, dtype=torch.float32), torch.tensor(y_c, dtype=torch.float32)

### START CODE HERE ###

### END CODE HERE ###

print("accuracy on the circles:", acc_circles)
assert acc_circles > 0.95, "3 hidden neurons are enough to surround the inner circle"
print("Exercise validated!")
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_boundary(model_moons, X_m, y_m, f"moons: {acc_moons:.3f}", axes[0])
plot_boundary(model_circles, X_c, y_c, f"circles: {acc_circles:.3f}", axes[1])
plt.show()

**What we see.** On the circles, the three hidden neurons draw three lines, and the output neuron keeps the
region inside all three: a rounded triangle around the inner circle. The boundary is no longer a line.

# Part D: how many hidden neurons?

### Exercise 17: a loop over the hidden size

For each `h` in `[1, 2, 4, 16]`, create a 2-h-1 network (after `torch.manual_seed(0)`), train it on the
moons (`lr=5.0`, 2000 epochs) and store it in the dictionary `models[h]` and its accuracy in `accs[h]`.

In [ ]:
models, accs = {}, {}
for h in [1, 2, 4, 16]:
    ### START CODE HERE ###
    pass  # replace this line
    ### END CODE HERE ###

print(accs)
assert set(accs) == {1, 2, 4, 16}, "one accuracy per hidden size"
assert accs[1] < 0.9, "with 1 hidden neuron the boundary is still (almost) a line"
assert accs[16] > 0.95 and accs[4] > 0.95, "4 and 16 hidden neurons follow the moons"
print("Exercise validated!")
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, h in zip(axes, [1, 2, 4, 16]):
    plot_boundary(models[h], X_m, y_m, f"{h} hidden: {accs[h]:.3f}", ax)
plt.show()

**What we see.** One hidden neuron is logistic regression in disguise: a single line, about the accuracy of
session 2. Two neurons can bend the boundary once. From 4 neurons on, the boundary follows the moons. More
neurons give more flexibility, and also more risk of fitting the noise of the training set: we need a test
set to know (Going further, and session 4).

# Going further

### Exercise 18: two hidden layers

Build `deep_model`: `Linear(2, 8)`, `Sigmoid`, `Linear(8, 8)`, `Sigmoid`, `Linear(8, 1)` (after
`torch.manual_seed(0)`), train it on the circles (`lr=5.0`, 2000 epochs), and compute `acc_deep`. Count its
parameters in `n_deep`.

In [ ]:
### START CODE HERE ###

### END CODE HERE ###

print("parameters:", n_deep, " accuracy:", acc_deep)
assert n_deep == 8 * 3 + 8 * 9 + 9, "layer by layer: n_out * (n_in + 1)"
assert acc_deep > 0.95, "the deeper network also separates the circles"
print("Exercise validated!")

### Exercise 19: train and test

Training accuracy says how well the model fits the examples it has seen. To know how it does on **new**
examples, keep some of them aside. With noisier moons (`noise=0.3`, 400 points), split 70 % / 30 % with
`train_test_split(X, y, test_size=0.3, random_state=0)`, train a 2-16-1 network on the training part only
(after `torch.manual_seed(0)`, `lr=5.0`, 2000 epochs) and compute `acc_train` and `acc_test`.

In [ ]:
from sklearn.model_selection import train_test_split

X_n, y_n = make_moons(n_samples=400, noise=0.3, random_state=0)
### START CODE HERE ###

### END CODE HERE ###

print("train accuracy:", round(acc_train, 3), " test accuracy:", round(acc_test, 3))
assert acc_test > 0.8, "the network should still classify new points well"
assert acc_train > 0.8, "the network should fit the training points well"
print("Exercise validated!")

**What we see.** The test accuracy is the number to report: it estimates the performance on new data. Here
it is even a little higher than the training accuracy: with only 120 test points, it is a noisy estimate,
and the noisy moons put some points on the wrong side whatever the model. A test accuracy far below the
training accuracy would be a sign of overfitting; we will monitor this gap from session 4 on.